# Rapor Üretici — IEEE benzeri .docx (Türkçe)

Bu notebook **Colab'da** çalışır ve bitmiş raporu (`MustafaKeremCekici_231307121.docx`)
`DATA_ROOT`'a (Drive) yazar. Figürler `DATA_ROOT/figures/`'tan okunur; eksik 2 figür
(dağılım + co-occurrence) burada yeniden üretilir. Tüm metin, tablo ve sayılar gömülüdür.

Çalıştır → Drive'dan `.docx`'i indir.

## 0) Kurulum

In [ ]:
!pip -q install python-docx
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/film-genre-project-data')
FIG = DATA_ROOT / 'figures'
print('figures var mi:', FIG.exists(), '| labels_v2:', (DATA_ROOT/'labels_v2.csv').exists())

## 1) Eksik 2 figürü yeniden üret (dağılım + co-occurrence)

In [ ]:
import numpy as np, csv
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

GEN = ['Action','Adventure','Animation','Comedy','Crime','Documentary','Drama','Family',
       'Fantasy','History','Horror','Mystery','Romance','Science Fiction','Thriller']
V1 = {'History':858,'Mystery':1168,'Animation':1266,'Fantasy':1334,'Science Fiction':1466,
      'Family':1698,'Horror':2004,'Adventure':2017,'Documentary':2017,'Crime':2128,
      'Romance':2269,'Action':2615,'Thriller':2948,'Comedy':5116,'Drama':6381}
rows = list(csv.DictReader(open(DATA_ROOT/'labels_v2.csv', encoding='utf-8')))
V2 = Counter(g for r in rows for g in r['genres'].split('|') if g in GEN)

order = sorted(GEN, key=lambda g: V2.get(g,0)); y=np.arange(len(order)); h=0.4
fig,ax=plt.subplots(figsize=(10,7))
ax.barh(y+h/2,[V1.get(g,0) for g in order],height=h,label='v1 ham (n=16307)')
ax.barh(y-h/2,[V2.get(g,0) for g in order],height=h,label='v2 dengeli (n=%d)'%len(rows))
ax.set_yticks(y); ax.set_yticklabels(order); ax.legend(); ax.set_xlabel('film sayısı')
ax.set_title('Tür dağılımı: çekim öncesi (v1) vs sonrası (v2)')
plt.tight_layout(); fig.savefig(FIG/'dist_before_after.png',dpi=120); plt.close()

idx={g:i for i,g in enumerate(GEN)}; M=np.zeros((15,15),int)
for r in rows:
    gs=[g for g in r['genres'].split('|') if g in GEN]
    for a in gs:
        for b in gs: M[idx[a],idx[b]]+=1
fig,ax=plt.subplots(figsize=(9,8)); im=ax.imshow(M,cmap='viridis'); fig.colorbar(im)
ax.set_xticks(range(15)); ax.set_xticklabels(GEN,rotation=60,ha='right',fontsize=8)
ax.set_yticks(range(15)); ax.set_yticklabels(GEN,fontsize=8)
ax.set_title('Tür birlikte-geçiş (co-occurrence) matrisi - v2')
plt.tight_layout(); fig.savefig(FIG/'cooccurrence_v2.png',dpi=120); plt.close()
print('2 figür üretildi ->', FIG)

## 2) .docx oluştur

In [ ]:
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()
def H(t, lvl=1): doc.add_heading(t, level=lvl)
def P(t): return doc.add_paragraph(t)
def fig(name, w=6.3, cap=''):
    p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(str(FIG/name), width=Inches(w))
    c=doc.add_paragraph(cap); c.alignment=WD_ALIGN_PARAGRAPH.CENTER
    for r in c.runs: r.italic=True; r.font.size=Pt(9)
def table(headers, data, cap=''):
    if cap:
        c=doc.add_paragraph(cap)
        for r in c.runs: r.italic=True; r.font.size=Pt(9)
    t=doc.add_table(rows=1, cols=len(headers)); t.style='Table Grid'
    for j,htxt in enumerate(headers):
        run=t.rows[0].cells[j].paragraphs[0].add_run(htxt); run.bold=True; run.font.size=Pt(9)
    for row in data:
        cs=t.add_row().cells
        for j,v in enumerate(row):
            run=cs[j].paragraphs[0].add_run(str(v)); run.font.size=Pt(9)

doc.add_heading('Film Afişlerinden Çoklu-Etiketli Tür Tahmini: Beş Görsel Transformatör Modelinin Karşılaştırmalı Değerlendirmesi', 0)
ap=P('Mustafa Kerem Çekici — 231307121 — Bilişim Sistemleri Mühendisliği, Kocaeli Üniversitesi — keremcek70@gmail.com')
ap.alignment=WD_ALIGN_PARAGRAPH.CENTER

H('Özet', 1)
P('Bu çalışmada film afişi görselinden filmin türlerinin tahmin edilmesi çoklu-etiketli bir '
  'sınıflandırma problemi olarak ele alınmıştır. TMDB resmî API kullanılarak 15 hedef tür için '
  'eksiklik-güdümlü bir örnekleme algoritmasıyla dengeli ve kombinasyon-farkında bir veri kümesi '
  '(23.640 film) oluşturulmuştur. Beş farklı görsel transformatör (ViT, DeiT, BeiT, Swin, CvT) '
  'ImageNet ön-eğitiminden ince ayarlanmış ve 5-katlı çapraz doğrulama ile değerlendirilmiştir. '
  'En iyi sonucu Swin vermiştir (makro F1=0,562; makro AUC=0,862); bu, ön-eğitimli CNN referansına '
  '(0,383) göre +0,179 ve şans seviyesindeki rasgele tahmin ediciye (0,149) göre yaklaşık 3,8 kat '
  'iyileşmedir. Dengeli veri sayesinde önceki aşırı tahmin sorunu giderilmiş (film başına ortalama '
  '2,4 tür, gerçek 2,18) ve frekans biası ortadan kalkmıştır.')
kp=P('Anahtar Kelimeler: çoklu-etiket sınıflandırma, görsel transformatör, film afişi, transfer öğrenme, çapraz doğrulama, TMDB')
for r in kp.runs: r.italic=True

H('1. Giriş', 1)
P('Film afişleri türü sezdiren güçlü görsel kodlar içerir (korku afişlerindeki karanlık tonlar, '
  'animasyonun çizgi-film estetiği gibi). Amaç, yalnızca afiş görselini girdi alarak filmin türlerini '
  'tahmin etmektir; bir film birden çok türe ait olabildiğinden problem çoklu-etiketlidir. İlk sürümde '
  '(v1) sıfırdan CNN ve dondurulmuş MobileNetV3 kullanılmış, ancak dengesiz veri yüzünden model sınıf '
  'frekanslarını ezberleyip film başına 5–7 tür tahmin ederek aşırı tahmin yapmıştır. Bu rapor problemi '
  'yeniden ele alan v2 sürümünü sunar: (i) dengeli ve kombinasyon-farkında veri toplama, (ii) beş '
  'transformatörün aynı koşulda karşılaştırılması, (iii) aşırı tahmin ve frekans biasının nicel '
  'giderildiğinin ve modelin gerçek görsel öğrenme yaptığının kanıtlanması.')

H('2. Veri Toplama ve Temizleme', 1)
H('2.1 HTML Kazımadan Resmî API’ye Geçiş', 2)
P('İlk yaklaşımda themoviedb.org HTML sayfaları kazınmış; site agresif hız sınırlaması uyguladığından '
  'TLS taklidi ve VPN’e rağmen sürekli HTTP 429 alınmıştır (tek oturumda 9.619 adet 429, 864 film '
  'kaybı; pratik hız 3 saatte ~500 film). Bu nedenle TMDB resmî JSON API’sine geçilmiştir: ücretsiz, '
  'günlük limitsiz, ~40–50 istek/sn. /discover/movie uç noktası her filmin tür listesini ve afiş yolunu '
  'tek istekte döndürdüğünden çoklu-etiketler ek maliyetsiz alınmış, afişler yalnızca kabul edilen '
  'filmler için indirilmiştir.')
H('2.2 Dengeli ve Kombinasyon-Farkında Örnekleme', 2)
P('v1 ileri derecede dengesizdi (Drama 6.381, History 858; ~7,4 kat). İki aşamalı yöntem kullanıldı: '
  '(1) Aday havuzu — her tür için vote_count≥10 olan adaylar (kalitesiz/postersiz elenir). '
  '(2) Eksiklik-güdümlü açgözlü seçim — her adımda en eksik tür, en az tür taşıyan filmle doldurulur; '
  'baskın türler (Drama, Comedy) “yolcu” olarak en sona bırakılır. vote_count eşiği 30 alındığında '
  'History 2.149’da kalıyordu; 10’a düşürülünce tüm nadir türler 3.000 hedefine ulaştı.')
H('2.3 Sonuç Veri Kümesi', 2)
P('Nihai veri kümesi 23.640 film içerir (film başına ort. 2,18 tür; afiş kapsamı yüzde 100). '
  'Dengesizlik 7,4 kattan 2,22 kata düşürülmüştür (History 858 → 3.000). Çekim öncesi/sonrası dağılım '
  'Şekil 1’de, birlikte-geçiş matrisi Şekil 2’de verilmiştir.')
fig('dist_before_after.png', 5.5, 'Şekil 1. Tür dağılımı: çekim öncesi (v1) vs sonrası (v2).')
fig('cooccurrence_v2.png', 5.0, 'Şekil 2. v2 veri kümesinde türler arası birlikte-geçiş matrisi.')

H('3. Yöntem', 1)
H('3.1 Modeller', 2)
P('Beş görsel transformatör ImageNet ön-eğitimli ağırlıklardan ince ayarlanmıştır: ViT (saf öz-dikkat), '
  'DeiT (damıtma ile veri-verimli ViT), BeiT (maskeli görüntü modellemesi), Swin (kaydırmalı pencereli '
  'hiyerarşik) ve CvT (evrişimle zenginleştirilmiş). Tümü base ölçekte; CvT-13 daha küçük kapasitelidir. '
  'HuggingFace Transformers ile, sınıflandırıcı başı 15 türe uyarlanarak '
  '(problem_type=multi_label_classification) yüklenmiştir.')
H('3.2 Ön İşleme ve Eğitim', 2)
P('Afişler 224×224’e yeniden boyutlandırılmış (kareye sıkıştırma), modelin kendi ortalama/standart '
  'sapma değerleriyle normalize edilmiş, eğitimde hafif renk artırımı uygulanmıştır. Kayıp '
  'BCEWithLogitsLoss; veri dengeli olduğundan ağır pos_weight kullanılmamıştır. Optimizasyon AdamW '
  '(lr=5e-5, weight_decay=0,01), kosinüs ısınma, karma duyarlık (AMP); toplu boyut 64, 5 epoch.')
H('3.3 Çapraz Doğrulama ve Değerlendirme', 2)
P('Veri, çoklu-etiket dağılımını koruyan MultilabelStratifiedKFold ile 5 kata bölünmüştür (kat başına '
  '~18.900 eğitim / ~4.730 doğrulama; örtüşme sıfır, mükemmel katmanlaşma). Her örnek bir kez doğrulama '
  'katında bulunduğundan kat-dışı (OOF) tahminler birleştirilip tüm metrikler bunlar üzerinden '
  'hesaplanmıştır. Sınıf başına eşik F1’i en üst düzeye çıkaracak biçimde optimize edilmiştir. Accuracy, '
  'Precision, Recall (Sensitivity), Specificity, F-Score ve AUC sınıf başına hesaplanıp makro/mikro '
  'ortalanmıştır.')

H('4. Deneysel Sonuçlar', 1)
P('Tablo 1 beş modeli v1 ve rasgele referanslarla karşılaştırır. En iyi model Swin’dir; beş '
  'transformatörün tamamı v1 referansını (0,383) açık biçimde geçmiştir.')
table(['Model','Makro F1','Mikro F1','Makro AUC','Prec.','Rec.'],
      [['Swin','0,562','0,566','0,862','0,541','0,587'],
       ['BeiT','0,534','0,539','0,844','0,517','0,555'],
       ['ViT','0,527','0,535','0,841','0,512','0,545'],
       ['DeiT','0,524','0,530','0,836','0,519','0,532'],
       ['CvT','0,501','0,509','0,823','0,474','0,536'],
       ['Frozen baseline (v1)','0,383','-','-','-','-'],
       ['Scratch CNN (v1)','0,333','-','-','-','-'],
       ['Rasgele (~2,4/film)','0,149','-','-','-','-'],
       ['Frekans-öncül','0,146','-','-','-','-']],
      'Tablo 1. Model karşılaştırması (OOF, eşik-optimize).')
fig('per_class_f1_models.png', 6.5, 'Şekil 3. Sınıf başına F1 (beş model).')

P('Tablo 2 Swin modelinin sınıf bazlı tüm metriklerini gösterir. En yüksek başarı Animation türündedir '
  '(F1=0,863; AUC=0,980); en düşükler görsel olarak en belirsiz türlerdedir (Fantasy, Mystery, Crime). '
  'Yüksek özgüllük (0,78–0,99) yanlış pozitiflerin azlığını, yani aşırı tahminin olmadığını doğrular.')
pc=[['Action','0,874','0,596','0,643','0,917','0,619','0,890'],
    ['Adventure','0,850','0,422','0,478','0,905','0,448','0,821'],
    ['Animation','0,966','0,896','0,833','0,986','0,863','0,980'],
    ['Comedy','0,869','0,620','0,669','0,912','0,644','0,887'],
    ['Crime','0,847','0,411','0,473','0,901','0,440','0,816'],
    ['Documentary','0,893','0,573','0,614','0,933','0,593','0,901'],
    ['Drama','0,753','0,550','0,674','0,784','0,606','0,811'],
    ['Family','0,905','0,613','0,681','0,938','0,645','0,917'],
    ['Fantasy','0,850','0,413','0,426','0,912','0,419','0,782'],
    ['History','0,863','0,461','0,482','0,918','0,471','0,841'],
    ['Horror','0,901','0,601','0,662','0,936','0,630','0,910'],
    ['Mystery','0,848','0,412','0,454','0,906','0,432','0,803'],
    ['Romance','0,875','0,506','0,563','0,920','0,533','0,866'],
    ['Science Fiction','0,889','0,564','0,559','0,937','0,561','0,868'],
    ['Thriller','0,827','0,475','0,590','0,873','0,526','0,844'],
    ['MAKRO','0,867','0,541','0,587','0,912','0,562','0,862']]
table(['Tür','Acc','Prec','Rec/Sens','Spec','F1','AUC'], pc,
      'Tablo 2. Swin — sınıf başına metrikler (OOF, eşik-optimize).')
fig('confusion_swin.png', 6.0, 'Şekil 4. Swin — sınıf başına karmaşıklık matrisleri.')
fig('roc_swin.png', 5.0, 'Şekil 5. Swin — ROC eğrileri (one-vs-rest), makro AUC=0,862.')
fig('loss_curves.png', 6.5, 'Şekil 6. Kat başına eğitim/doğrulama kayıp eğrileri (her model).')

P('Eğitim/çıkarım süreleri Tablo 3’te özetlenmiştir. Yüksek kapasiteli modeller 2–3. epoch’tan sonra '
  'aşırı öğrenirken CvT 5 epoch’ta hâlâ yetersiz öğrenme bölgesindedir (kapasite/veri dengesi). En '
  'iyi-F1 kontrol noktası saklandığından aşırı öğrenme son modeli etkilememiştir.')
table(['Model','Eğitim (dk/kat)','Çıkarım (ms/görsel)','Ort. tür/film'],
      [['ViT','3,94','1,58','2,36'],['DeiT','3,96','1,58','2,29'],['BeiT','4,16','1,60','2,39'],
       ['Swin','5,84','1,63','2,41'],['CvT','4,73','1,58','2,52']],
      'Tablo 3. Eğitim/çıkarım süresi ve ortalama tahmin sayısı.')

P('Kritik kontroller: (i) Aşırı tahmin giderildi — film başına ort. 2,3–2,5 tür (gerçek 2,18), v1’de '
  '5–7 idi. (ii) Frekans biası giderildi — v1’de en iyi türler Drama/Comedy, en kötüler '
  'History/Documentary (~0,10) idi; v2’de en iyi tür Animation (0,863), History 0,10→0,47, Documentary '
  '0,11→0,59. (iii) Rasgele referans — ~2,4 tür rasgele tahmin makro F1=0,149; afişe bakmadan '
  'frekans-öncül tahmin 0,146. Swin’in 0,562’si şansın ~3,8 katıdır; sinyal afişten gelmektedir.')
fig('samples_swin.png', 6.0, 'Şekil 7. Swin — örnek afişler için gerçek ve tahmin edilen türler.')

H('5. Tartışma', 1)
P('Dengeli ve kombinasyon-farkında veri toplama belirleyici olmuştur: v1’deki aşırı tahmin ve frekans '
  'biası ağır kayıp ağırlıklandırmasından değil veri dengesizliğinden kaynaklanıyordu ve kaynağında '
  'çözülmüştür. Hiyerarşik Swin’in en iyi sonucu vermesi, görece küçük veride tümevarımsal ön-yargısı '
  'güçlü modellerin saf ViT’e üstün geldiği bulgularıyla tutarlıdır. CvT’nin düşük başarısı küçük '
  'kapasitesi ve yetersiz öğrenmesiyle açıklanır. Co-occurrence (etiket korelasyonu) riski: kombinasyon '
  'dengesine dikkat edilerek bu risk azaltılmış, nadir türlerdeki büyük artış riskin baskın olmadığını '
  'göstermiştir; ancak tamamen elenmiş değildir. Afişlerin kareye sıkıştırılması en/boy oranını bozar; '
  'en zayıf türler görsel olarak en belirsiz olanlardır.')

H('6. Sonuç', 1)
P('Afişten çoklu-etiketli tür tahmini için beş görsel transformatör aynı koşulda karşılaştırılmıştır. '
  'En iyi model Swin (makro F1=0,562) v1 referansına göre yaklaşık yüzde 47 görece iyileşme sağlamış; '
  'aşırı tahmin ve frekans biası nicel giderilmiş, rasgele referanslarla modelin görsel sinyali '
  'gerçekten öğrendiği kanıtlanmıştır. Gelecek çalışmalarda en/boy oranını koruyan ön işleme, daha güçlü '
  'düzenlileştirme ve metin (başlık) ile çok-modlu birleşim incelenebilir.')

H('Kaynaklar', 1)
refs=['A. Dosovitskiy et al., "An image is worth 16x16 words," ICLR, 2021.',
      'H. Touvron et al., "Training data-efficient image transformers & distillation through attention," ICML, 2021.',
      'H. Bao et al., "BEiT: BERT pre-training of image transformers," ICLR, 2022.',
      'Z. Liu et al., "Swin transformer," ICCV, 2021.',
      'H. Wu et al., "CvT: Introducing convolutions to vision transformers," ICCV, 2021.',
      'K. Sechidis et al., "On the stratification of multi-label data," ECML PKDD, 2011.',
      'T. Wolf et al., "Transformers: State-of-the-art NLP," EMNLP, 2020.',
      'The Movie Database (TMDB), API documentation, developer.themoviedb.org, 2026.']
for i,ref in enumerate(refs,1): P('[%d] %s' % (i, ref))

out = DATA_ROOT / 'MustafaKeremCekici_231307121.docx'
doc.save(str(out))
print('KAYDEDILDI ->', out)